# Position and Phase Transition Studies

In this notebook, we explore the fundamental **equations of motion** and compare the **numerical methods** used in our [MD simulation](). This is achieved by visualizing **particle configurations** in the system. Additionally, we employ the counting of **neighboring particles** throughout the simulation as an indicator of particle density and the current **phase of the system** (solid, liquid, or gas). 

As part of this study, we use the output data from the [MD code]() to analyze both **phase transitions** (using a thermostat to control temperature during annealing and heating processes) and other routine investigations.  
  
The visualization techniques employed include:
- **Snapshots** of particle positions at specific times.
- **Particle trajectories** over a specific time window, highlighting start and end positions.
- **Dynamic animations** to observe the effects of interactions on particle movements over time.
- **Neighbor count plots** over time, to identify critical points in phase transitions and determine the critical temperature during the annealing process.

---

### Outline:
1. **Many-Body Problem and Equations of Motion**: Overview of the many-body problem in classical mechanics and the application of Newton's laws of motion.
2. **Integration Methods**: A comparison of common numerical methods used to solve equations of motion in many-body systems:
   - **Euler Method**
   - **Symplectic Euler Method**
   - **Velocity Verlet Method**
3. **Phase Transition Studies**: Analysis of particle neighbors over time to investigate phase transitions.

---


## 1. Many-Body Problem and Equations of Motion

Molecular Dynamics (MD) simulations aim to track the precise **positions** and **velocities** of each particle in a system over time. This is a classical mechanics problem where, given the governing **forces** or the **potential energy** (the negative gradient of which gives the force), we can solve the equations of motion to determine the system's evolution at each time step. This includes key properties such as accelerations, velocities, positions, kinetic energy, temperature, and other physical observables.

### Degrees of Freedom and Equations of Motion
In a 2D system, each particle has two, or three degrees of freedom: **x-position**, **y-position**, and **orientation** $ \phi $. In our simulation, we use [`particleDot`]() for particles that can only have translational motion, and [`particleOriented`]() for the ones that can have orientation($\phi$).
Therefore, the motion must be described by the equations of motion for both translational and rotational dynamics.

Newton's second law describes the translational motion:

$$
\mathbf{F} = m \mathbf{a}, \quad \mathbf{a} = \frac{d\mathbf{v}}{dt}, \quad \mathbf{v} = \frac{d\mathbf{r}}{dt}
$$

where $ \mathbf{r} = (x, y) $ and $ \mathbf{F} = (F_x, F_y) $ are the position and force vectors in Cartesian coordinates.

For rotational motion, the angular acceleration is governed by the torque $ \tau $:

$$
\tau = I \alpha, \quad \alpha = \frac{d\omega}{dt}, \quad \omega = \frac{d\phi}{dt}
$$

where:
- $ \phi $ is the orientation angle,
- $ \omega $ is the angular velocity,
- $ \alpha $ is the angular acceleration,
- $ I $ is the moment of inertia.

### Force Components and Generalization
In our simulation, the force components can be divided into two parts:
- **Translational Force Components**: $ F_x $ and $ F_y $ are used to update the linear positions $x$ and $y$ and their respective velocities.
- **Rotational Force Component**: The tangential component of the force contributes to the **torque**, which is used to update the orientation $ \phi $ and angular velocity $ \omega $.

The force is derived from a potential function $ U(\mathbf{r}, \phi) $. The generalized force expressions for each degree of freedom are:

$$
F_x = -\frac{\partial U}{\partial x}, \quad
F_y = -\frac{\partial U}{\partial y}, \quad
\tau = -\frac{\partial U}{\partial \phi}
$$

### Generalized Approach in the Code
The simulation uses a **generalized format** for handling different types of particles:
- [`particleDot`](): Non-oriented particles, described only by translational degrees of freedom.
- [`particleOriented`](): Oriented particles with both translational and rotational dynamics.

Each particle type has its own potential and force calculations implemented in separate **acceleration functions**. However, the code employs a **generalized interface** using C++ templates and traits (template template) to ensure flexibility in setting and retrieving positions, velocities, and forces uniformly across different particle types.

This design allows for a unified treatment of both translational and rotational motions while maintaining the flexibility to choose different potentials and force calculations.

---



## 2. Numerical Integration Methods

Solving the equations of motion analytically becomes impractical for systems with more than few interacting particles. This complexity necessitates the use of computational and numerical methods, known as **numerical integration methods**, to approximate the solutions.

Several numerical integration methods are employed in MD simulations to solve the equations of motion:

1. **Euler Method**:
   - **Description**: A straightforward method based on the Taylor expansion, developed by Leonhard Euler.
   - **Formulation**:
     $$ x(t + \Delta t) = x(t) + v(t) \cdot \Delta t $$
     $$ v(t + \Delta t) = v(t) + a(t) \cdot \Delta t $$
   - **Pros and Cons**: Easy to implement but less accurate and not symplectic, which can lead to energy drift over time.

2. **Symplectic Euler Method** (also known as Semi-Implicit Euler or Euler-Cromer):
   - **Description**: An improved version of the Euler method that is symplectic, meaning it preserves the geometric properties of Hamiltonian systems, leading to better energy conservation.
   - **Formulation**:
     $$ v(t + \Delta t) = v(t) + a(t) \cdot \Delta t $$
     $$ x(t + \Delta t) = x(t) + v(t + \Delta t) \cdot \Delta t $$
   - **Pros and Cons**: More accurate than the standard Euler method and preserves symplectic structure, but still only first-order accurate.

3. **Velocity Verlet Method**:
   - **Description**: A widely used method in MD simulations that offers a good balance between accuracy and computational efficiency.
   - **Formulation**:
     $$ x(t + \Delta t) = x(t) + v(t) \cdot \Delta t + \frac{1}{2} \cdot a(t) \cdot (\Delta t)^2 $$
     $$ v(t + \Delta t) = v(t) + \frac{1}{2} \cdot [a(t) + a(t + \Delta t)] \cdot \Delta t $$
   - **Pros and Cons**: Second-order accurate, symplectic, and time-reversible, making it suitable for long-term simulations.

In computational methods, there's always a trade-off between accuracy and computational cost. While higher-order methods like Runge-Kutta offer greater accuracy, they come with increased computational expense. The **Velocity Verlet** method provides a favorable balance, making it well-suited for our MD simulations.

In our [MD simulation](), we've implemented the **Euler**, **Symplectic Euler**, and **Velocity Verlet** methods. Users can select the desired integration method as a runtime parameter to observe and compare the outcomes of each approach.

---
 